# Upload data and test Athena

Uploads the batches from `data/landing/` to `s3://<lake bucket>/landing/`, checks the upload against the local files,
then runs the first queries in Athena.
Run the cells top to bottom. Each cell is safe to run twice, and prints the AWS CLI command it runs (`$ aws ...`).

## 1. Setup

In [1]:
import sys
from pathlib import Path

# Works whether Jupyter starts in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "infra"))

# aws() runs the AWS CLI and prints the command and its output (see infra/aws_helpers.py)
from aws_helpers import aws, aws_json, run_athena, whoami, LAKE, RESULTS, SETTINGS

LANDING = ROOT / "data" / "landing"
N_BATCHES = SETTINGS.get("number_of_batches", 4)

print("identity:", whoami())
print("source  :", LANDING)
print("target  : s3://" + LAKE + "/landing/")

identity: user/Ahmed
source  : D:\Visual Code\AWS_endtoend_project\data\landing
target  : s3://patientflow-datalake-zhmkbf/landing/


## 2. Check the local batches
Stop before uploading if the batches are missing, empty or incomplete.

In [2]:
local = {}      # batch name -> {file name: size in bytes}
for batch_dir in sorted(LANDING.glob("batch_*")):
    local[batch_dir.name] = {f.name: f.stat().st_size for f in batch_dir.iterdir() if f.is_file()}

assert len(local) == N_BATCHES, f"expected {N_BATCHES} batches in {LANDING}, found {len(local)}"
for batch, files in local.items():
    assert files, f"{batch} is empty"
    assert all(name.endswith(".json") for name in files), f"{batch} has a file that is not .json"
    assert all(size > 0 for size in files.values()), f"{batch} has an empty file"

reference = [n for n in local["batch_01"] if n.startswith(("hospitalInformation", "practitionerInformation"))]
assert len(reference) == 2, "batch_01 must contain the hospital and practitioner files"

total_files = sum(len(f) for f in local.values())
total_mb = sum(sum(f.values()) for f in local.values()) / 1e6
print(f"OK  {len(local)} batches, {total_files} files, {total_mb:,.1f} MB")

OK  4 batches, 139 files, 294.9 MB


## 3. Upload
`aws s3 sync` copies a folder to S3 and **skips files that are already there** with the same size and date,
so running it again only uploads what changed. `--only-show-errors` hides the per-file lines (there are hundreds).

In [3]:
aws("s3", "sync", "data/landing", f"s3://{LAKE}/landing/", "--only-show-errors")

$ aws s3 sync data/landing s3://patientflow-datalake-zhmkbf/landing/ --only-show-errors


<aws exit code 0>

## 4. Verify the upload
List what is in S3 and compare it with the local files, batch by batch: same files, same sizes.

In [4]:
objects = aws_json("s3api", "list-objects-v2", "--bucket", LAKE, "--prefix", "landing/")["Contents"]

remote = {}     # batch name -> {file name: size in bytes}
for obj in objects:
    _, batch, name = obj["Key"].split("/", 2)       # landing/batch_01/file.json
    remote[batch] = {**remote.get(batch, {}), name: obj["Size"]}

assert remote == local, "S3 does not match the local batches"
print(f"{'batch':<10}{'files':>7}{'MB':>9}")
for batch, files in remote.items():
    print(f"{batch:<10}{len(files):>7}{sum(files.values()) / 1e6:>9.1f}")
print(f"\nOK  S3 matches the local files: {len(objects)} objects, {sum(o['Size'] for o in objects) / 1e6:,.1f} MB")

batch       files       MB
batch_01       37    110.8
batch_02       34     45.8
batch_03       34     67.1
batch_04       34     71.3

OK  S3 matches the local files: 139 objects, 294.9 MB


In [5]:
# Default encryption should have been applied to the uploaded objects
head = aws_json("s3api", "head-object", "--bucket", LAKE, "--key", objects[0]["Key"])
assert head["ServerSideEncryption"] == "AES256"
print("OK  uploaded objects are encrypted:", head["ServerSideEncryption"])

OK  uploaded objects are encrypted: AES256


## 5. Test Athena
`run_athena` (in `infra/aws_helpers.py`) starts a query in the `pf-wg` workgroup, waits for it to finish and returns the rows.
Athena is asynchronous: `start-query-execution` returns an id, then we poll `get-query-execution` until the query is `SUCCEEDED`.

In [6]:
test = run_athena("SELECT 1 AS test")
print("query id    :", test["query_id"])
print("result      :", test["header"], test["rows"])
print("bytes scanned:", test["scanned_bytes"])
assert test["rows"] == [["1"]]
print("OK  Athena runs queries in the workgroup")

query id    : bad50251-879a-4243-b083-9ccef010846b
result      : ['test'] [['1']]
bytes scanned: 0
OK  Athena runs queries in the workgroup


In [7]:
databases = run_athena("SHOW DATABASES")
names = {row[0] for row in databases["rows"]}
print("databases visible to Athena:", sorted(names))
for name in SETTINGS["glue_databases"].values():
    assert name in names, f"{name} is not visible to Athena"
print("OK  Athena sees the Glue databases")

databases visible to Athena: ['pf_gold', 'pf_silver']
OK  Athena sees the Glue databases


In [8]:
# Query results are saved in the results bucket (the workgroup enforces this location)
saved = aws_json("s3api", "list-objects-v2", "--bucket", RESULTS, "--prefix", test["query_id"])["Contents"]
print("result files for the test query:", [o["Key"] for o in saved])
assert saved
print("OK  results are written to the results bucket")

result files for the test query: ['bad50251-879a-4243-b083-9ccef010846b.csv', 'bad50251-879a-4243-b083-9ccef010846b.csv.metadata']
OK  results are written to the results bucket


## What the uploaded data is for

| Location | Used by |
|---|---|
| `landing/batch_01 ... batch_10` | The ingest Lambda copies one batch per run into `bronze/patient_records/ingest_date=<date>/` (Phase 4) |

Batch `N` arrives on `2026-11-01 + (N - 1)` days. Uploading everything once simulates a source system that already delivered the files.

This completes Phase 3. **Next:** Phase 4, the ingest Lambda.